## Generating Fact_Encounter Table 

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Setup
np.random.seed(42)
n_rows = 100000
n_patients = 20000 

# 2. Expanded Cardinality Logic
data = {
    'Encounter_SK': range(100001, 100001 + n_rows),
    'Patient_FK': np.random.randint(5001, 5001 + n_patients, size=n_rows),
    'Hospital_FK': np.random.randint(201, 216, size=n_rows),       # 15 Hospitals
    'Diagnosis_FK': np.random.randint(301, 351, size=n_rows),      # 50 Diagnosis Groups
    'Provider_FK': np.random.randint(701, 1201, size=n_rows),      # 500 Providers
    'Payer_FK': np.random.randint(901, 911, size=n_rows),         # 10 Payers
    'Quality_Flag_FK': np.random.choice([801, 802, 803, 804, 805], size=n_rows, p=[0.85, 0.04, 0.04, 0.04, 0.03]),
    'Acuity_Score': np.random.choice([1, 2, 3, 4, 5], size=n_rows, p=[0.1, 0.2, 0.4, 0.2, 0.1])
}

df = pd.DataFrame(data)

# 3. Clinical Logic: Length_of_Stay (LOS)
# Heart Failure (let's assume 301-305 are cardiac) and high acuity stay longer
df['Length_of_Stay'] = np.random.poisson(lam=3, size=n_rows) + 1
df.loc[df['Diagnosis_FK'] <= 305, 'Length_of_Stay'] += 2
df.loc[df['Acuity_Score'] >= 4, 'Length_of_Stay'] += 1

# 4. Target Variable: Readmission_30D Logic
# Increased risk for cardiac codes, high acuity, and specific hospitals (simulating poor site performance)
readmit_prob = np.zeros(n_rows)
readmit_prob += np.where(df['Diagnosis_FK'] <= 305, 0.30, 0)
readmit_prob += np.where(df['Acuity_Score'] == 5, 0.40, 0)
readmit_prob += np.where(df['Hospital_FK'] == 215, 0.20, 0) # Hospital 215 is a "poor performer"
readmit_prob += np.random.normal(0, 0.1, n_rows) 

df['Readmission_30D'] = (readmit_prob > 0.50).astype(int)

# 5. Outlier Insertion (0.5% records)
outlier_idx = df.sample(frac=0.005).index
df.loc[outlier_idx, 'Length_of_Stay'] = np.random.randint(15, 40, size=len(outlier_idx))
df.loc[outlier_idx, 'Total_Charge'] = np.random.randint(70000, 150000, size=len(outlier_idx))

# 6. Normal Total_Charge Calculation
mask = ~df.index.isin(outlier_idx)
df.loc[mask, 'Total_Charge'] = (df.loc[mask, 'Length_of_Stay'] * 3100) + \
                                (df.loc[mask, 'Acuity_Score'] * 750) + \
                                np.random.normal(300, 100, sum(mask))

# 7. Date Handling
start_date = datetime(2026, 1, 1)
df['temp_admit'] = [start_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n_rows)]
df['temp_disch'] = df.apply(lambda x: x['temp_admit'] + timedelta(days=x['Length_of_Stay']), axis=1)

df['Admission_Date_FK'] = df['temp_admit'].dt.strftime('%Y%m%d').astype(int)
df['Discharge_Date_FK'] = df['temp_disch'].dt.strftime('%Y%m%d').astype(int)

# 8. Final Column Order
ordered_cols = [
    'Encounter_SK', 'Patient_FK', 'Admission_Date_FK', 'Discharge_Date_FK',
    'Hospital_FK', 'Diagnosis_FK', 'Provider_FK', 'Payer_FK', 'Quality_Flag_FK',
    'Length_of_Stay', 'Total_Charge', 'Readmission_30D', 'Acuity_Score'
]

# Export to Excel (Note: Requires 'openpyxl' installed)
df[ordered_cols].to_excel('Fact_Encounters.xlsx', index=False, engine='openpyxl')
print("Finalized Fact_Encounters_100k.xlsx has been generated.")

Finalized Fact_Encounters_100k.xlsx has been generated.


## Generating Dim_Diagnosis Table

In [ ]:
import pandas as pd
import numpy as np

# 1. Clinical Data Definitions (USA 2025 Context)
# Grouping 50 codes into 5 Major Diagnostic Categories (MDCs)
clinical_map = [
    # MDC: Circulatory (High Readmission Risk)
    {'MDC': 'Circulatory', 'CCS': 'Heart Failure', 'Weight': 3.8, 'Chronic': 1, 'Start': 301, 'End': 305, 'ICD': 'I50.'},
    # MDC: Respiratory
    {'MDC': 'Respiratory', 'CCS': 'COPD and Bronchitis', 'Weight': 3.2, 'Chronic': 1, 'Start': 306, 'End': 315, 'ICD': 'J44.'},
    # MDC: Endocrine
    {'MDC': 'Endocrine', 'CCS': 'Diabetes mellitus', 'Weight': 2.5, 'Chronic': 1, 'Start': 316, 'End': 325, 'ICD': 'E11.'},
    # MDC: Digestive
    {'MDC': 'Digestive', 'CCS': 'Gastrointestinal hemorrhage', 'Weight': 2.1, 'Chronic': 0, 'Start': 326, 'End': 335, 'ICD': 'K92.'},
    # MDC: Injury/Other (Lower Risk)
    {'MDC': 'Injury', 'CCS': 'Complication of device', 'Weight': 1.8, 'Chronic': 0, 'Start': 336, 'End': 350, 'ICD': 'T81.'}
]

diagnosis_records = []

for group in clinical_map:
    for sk in range(group['Start'], group['End'] + 1):
        suffix = sk % 10 # Creating unique sub-codes
        diagnosis_records.append({
            'Diagnosis_SK': sk,
            'ICD10_Code': f"{group['ICD']}{suffix}",
            'Diagnosis_Description': f"{group['CCS']} - Type {suffix}",
            'CCS_Category': group['CCS'],
            'Major_Diagnostic_Cat': group['MDC'],
            'Comorbidity_Weight': group['Weight'] + np.random.uniform(-0.5, 0.5),
            'Is_Chronic': group['Chronic']
        })

# 2. Build and Export
df_diag = pd.DataFrame(diagnosis_records)
df_diag.to_excel('Dim_Diagnosis.xlsx', index=False, engine='openpyxl')

print(f"Success! Dim_Diagnosis_50.xlsx generated with clinical mapping for {len(df_diag)} codes.")

## Generating Dim_Patients

In [ ]:
import pandas as pd
import numpy as np

# 1. Setup
np.random.seed(42)
n_patients = 20000

# 2. Key Generation (Guaranteed Unique EHR IDs)
patient_sks = np.arange(5001, 5001 + n_patients)
# Generate a pool of 30,000 unique IDs and sample 20,000 to ensure 0 duplicates
unique_ids = np.random.choice(np.arange(100000, 999999), size=n_patients, replace=False)
ehr_ids = [f'US-{x}' for x in unique_ids]

# 3. Demographic Logic
genders = np.random.choice(['Male', 'Female', 'Other'], size=n_patients, p=[0.48, 0.50, 0.02])

# Skewed Age for USA 2025 Readmission Context
seniors = np.random.normal(74, 10, int(n_patients * 0.65))
adults = np.random.randint(18, 64, int(n_patients * 0.35))
all_ages = np.concatenate([seniors, adults])
np.random.shuffle(all_ages)
ages = np.clip(all_ages, 18, 99).astype(int)

# 4. Expanded Geography (50 Unique Zip Prefixes)
# Mix of major hubs and surrounding rural prefixes
zip_prefixes = [
    '606', '100', '331', '902', '752', '191', '850', '303', '981', '021',
    '441', '303', '631', '482', '200', '328', '802', '554', '770', '941'
] + [str(x) for x in np.random.randint(100, 999, size=30)] # Add 30 random surrounding zips

# 5. Build Final Table
data = {
    'Patient_SK': patient_sks,
    'Original_EHR_ID': ehr_ids,
    'Gender': genders,
    'Age_at_Encounter': ages,
    'Race_Ethnicity': np.random.choice(
        ['Caucasian', 'African American', 'Hispanic', 'Asian', 'Native American'], 
        size=n_patients, p=[0.58, 0.14, 0.19, 0.07, 0.02]
    ),
    'Zip_Code_Prefix': np.random.choice(zip_prefixes, size=n_patients),
    'Primary_Language': np.random.choice(
        ['English', 'Spanish', 'Mandarin', 'Arabic', 'Other'], 
        size=n_patients, p=[0.78, 0.14, 0.04, 0.02, 0.02]
    ),
    'Marital_Status': np.random.choice(['Married', 'Single', 'Divorced', 'Widowed'], size=n_patients),
    'Is_Deceased': np.random.choice([0, 1], size=n_patients, p=[0.98, 0.02])
}

df_patients = pd.DataFrame(data)

# Save to Excel
df_patients.to_excel('Dim_Patients.xlsx', index=False, engine='openpyxl')
print("Dim_Patients_20k_Final.xlsx generated with unique IDs and expanded geography.")

## Generating Dim_Providers Table

In [3]:
import pandas as pd
import numpy as np

# 1. Setup
np.random.seed(42)
n_providers = 500

# 2. Key Generation (Using vectorized numpy calls for speed)
# Unique 10-digit NPIs
npi_pool = np.arange(1000000000, 1000000000 + n_providers, dtype=np.int64)

# 3. Attributes (Ensuring all lengths = 500)
specialties = [
    'Internal Medicine', 'Cardiology', 'Pulmonology', 'Endocrinology', 
    'General Surgery', 'Emergency Medicine', 'Neurology', 'Orthopedic Surgery',
    'Gastroenterology', 'Infectious Disease'
]

data = {
    'Provider_SK': np.arange(701, 701 + n_providers),
    'NPI_Number': npi_pool.astype(str),
    'Provider_Name': ["" for _ in range(n_providers)], # Blank as requested
    'Primary_Specialty': np.random.choice(specialties, size=n_providers),
    'Experience_Years': np.random.randint(1, 41, size=n_providers),
    'Provider_Type': np.random.choice(
        ['Physician', 'Nurse Practitioner', 'Physician Assistant', 'Resident'], 
        size=n_providers, p=[0.70, 0.15, 0.10, 0.05]
    ),
    'Is_In_Network': np.random.choice([0, 1], size=n_providers, p=[0.15, 0.85])
}

# 4. Create DataFrame
df_providers = pd.DataFrame(data)

# 5. Export to Excel
# Ensure you have 'openpyxl' installed: pip install openpyxl
try:
    df_providers.to_excel('Dim_Providers.xlsx', index=False, engine='openpyxl')
    print("Success! Dim_Providers.xlsx created in seconds.")
except Exception as e:
    print(f"Export Error: {e}. Try saving as CSV if Excel fails: df.to_csv('Dim_Providers.csv')")

Success! Dim_Providers_500_Final.xlsx created in seconds.


## Generating Dim_Hospitals Table

In [1]:
import pandas as pd
import numpy as np

# 1. Setup
n_hospitals = 15

# 2. Facility Definitions (USA 2025 Context)
hospitals = [
    {'Name': 'Mercy University Medical', 'Type': 'Teaching Hospital', 'City': 'Chicago', 'State': 'IL', 'Beds': 950, 'Trauma': 1},
    {'Name': 'Manhattan General', 'Type': 'Community Hospital', 'City': 'New York', 'State': 'NY', 'Beds': 420, 'Trauma': 2},
    {'Name': 'Miami Coastal Health', 'Type': 'Teaching Hospital', 'City': 'Miami', 'State': 'FL', 'Beds': 680, 'Trauma': 1},
    {'Name': 'Sonoran Valley Clinic', 'Type': 'Community Hospital', 'City': 'Phoenix', 'State': 'AZ', 'Beds': 305, 'Trauma': 3},
    {'Name': 'Lonestar Methodist', 'Type': 'Teaching Hospital', 'City': 'Dallas', 'State': 'TX', 'Beds': 550, 'Trauma': 1},
    {'Name': 'Liberty Square Medical', 'Type': 'Community Hospital', 'City': 'Philadelphia', 'State': 'PA', 'Beds': 390, 'Trauma': 2},
    {'Name': 'Pacific Bay Surgery Center', 'Type': 'Specialized Surgery Center', 'City': 'San Francisco', 'State': 'CA', 'Beds': 120, 'Trauma': 4},
    {'Name': 'Gateway Arch Community', 'Type': 'Community Hospital', 'City': 'St. Louis', 'State': 'MO', 'Beds': 210, 'Trauma': 3},
    {'Name': 'Peachtree Health Hub', 'Type': 'Teaching Hospital', 'City': 'Atlanta', 'State': 'GA', 'Beds': 480, 'Trauma': 1},
    {'Name': 'Evergreen Medical Center', 'Type': 'Community Hospital', 'City': 'Seattle', 'State': 'WA', 'Beds': 275, 'Trauma': 2},
    {'Name': 'Bunker Hill General', 'Type': 'Teaching Hospital', 'City': 'Boston', 'State': 'MA', 'Beds': 520, 'Trauma': 1},
    {'Name': 'Front Range Health', 'Type': 'Community Hospital', 'City': 'Denver', 'State': 'CO', 'Beds': 340, 'Trauma': 2},
    {'Name': 'Hoosier Community Hospital', 'Type': 'Community Hospital', 'City': 'Indianapolis', 'State': 'IN', 'Beds': 190, 'Trauma': 4},
    {'Name': 'Erie Shore Medical', 'Type': 'Community Hospital', 'City': 'Cleveland', 'State': 'OH', 'Beds': 245, 'Trauma': 3},
    {'Name': 'High Plains Rural Center', 'Type': 'Community Hospital', 'City': 'Scottsbluff', 'State': 'NE', 'Beds': 45, 'Trauma': 5}
]

# 3. Create DataFrame
data = {
    'Hospital_SK': range(201, 201 + n_hospitals),
    'Site_Name': [h['Name'] for h in hospitals],
    'City': [h['City'] for h in hospitals],
    'State_Province': [h['State'] for h in hospitals],
    'Facility_Type': [h['Type'] for h in hospitals],
    'Total_Bed_Count': [h['Beds'] for h in hospitals],
    'Trauma_Level': [h['Trauma'] for h in hospitals]
}

df_hospitals = pd.DataFrame(data)

# 4. Export to Excel
df_hospitals.to_excel('Dim_Hospitals.xlsx', index=False, engine='openpyxl')
print("Success! Dim_Hospitals_15.xlsx generated with 7 columns and trauma level logic.")

Success! Dim_Hospitals_15.xlsx generated with 7 columns and trauma level logic.


## Generating Dim_Payers Table

In [3]:
import pandas as pd
import numpy as np

# 1. Define Realistic Payer Data for USA 2025
payer_data = [
    {'Name': 'Medicare Part A', 'Type': 'Medicare', 'Plan': 'Traditional', 'Gov': 1, 'Rate': 0.88},
    {'Name': 'Medicare Advantage', 'Type': 'Medicare', 'Plan': 'HMO', 'Gov': 1, 'Rate': 0.84},
    {'Name': 'State Medicaid', 'Type': 'Medicaid', 'Plan': 'Managed Medicaid', 'Gov': 1, 'Rate': 0.72},
    {'Name': 'Blue Cross Blue Shield', 'Type': 'Private', 'Plan': 'PPO', 'Gov': 0, 'Rate': 0.95},
    {'Name': 'UnitedHealthcare', 'Type': 'Private', 'Plan': 'PPO', 'Gov': 0, 'Rate': 0.92},
    {'Name': 'Aetna Health', 'Type': 'Private', 'Plan': 'HMO', 'Gov': 0, 'Rate': 0.89},
    {'Name': 'Cigna Global', 'Type': 'Private', 'Plan': 'PPO', 'Gov': 0, 'Rate': 0.94},
    {'Name': 'Kaiser Permanente', 'Type': 'Private', 'Plan': 'HMO', 'Gov': 0, 'Rate': 0.87},
    {'Name': 'Self-Pay Indigent', 'Type': 'Self-Pay', 'Plan': 'None', 'Gov': 0, 'Rate': 0.15},
    {'Name': 'Self-Pay International', 'Type': 'Self-Pay', 'Plan': 'None', 'Gov': 0, 'Rate': 0.25}
]

# 2. Create DataFrame
data = {
    'Payer_SK': range(901, 901 + len(payer_data)),
    'Payer_Name': [p['Name'] for p in payer_data],
    'Payer_Type': [p['Type'] for p in payer_data],
    'Plan_Type': [p['Plan'] for p in payer_data],
    'Is_Government_Funded': [p['Gov'] for p in payer_data],
    'Reimbursement_Rate': [p['Rate'] for p in payer_data]
}

df_payers = pd.DataFrame(data)

# 3. Export to Excel
df_payers.to_excel('Dim_Payers.xlsx', index=False, engine='openpyxl')
print("Success! Dim_Payers_10.xlsx generated with financial and government-funded logic.")

Success! Dim_Payers_10.xlsx generated with financial and government-funded logic.


## Generating Dim_Date Table

In [2]:
import pandas as pd
import numpy as np

# 1. Generate every date in the year 2025
dates = pd.date_range(start='2024-01-01', end='2025-12-31')

# 2. Build the Dimension Attributes
dim_date_data = {
    # The Date_SK is the JOIN KEY (YYYYMMDD format)
    'Date_SK': dates.strftime('%Y%m%d').astype(int),
    'Full_Date': dates,
    'Day_of_Week': dates.day_name(),
    'Is_Weekend': np.where(dates.dayofweek >= 5, 1, 0),
    # Simple Holiday Logic for US 2025 (New Years, July 4, Dec 25)
    'Is_Holiday': np.where(dates.strftime('%m-%d').isin(['01-01', '07-04', '12-25']), 1, 0),
    'Month_Name': dates.month_name(),
    'Quarter': dates.quarter,
    'Fiscal_Year': dates.year
}

# 3. Create DataFrame
df_date = pd.DataFrame(dim_date_data)

# 4. Export to Excel
df_date.to_excel('Dim_Date.xlsx', index=False, engine='openpyxl')
print("Success! Dim_Date.xlsx generated with 365 unique dates.")

Success! Dim_Date.xlsx generated with 365 unique dates.


## Generating Dim_Quality_Flags Table

In [3]:
import pandas as pd

# 1. Defining the Logic for the 5 Quality Keys
# This structure maps specific attributes to your 801-805 keys
data = {
    'Quality_Flag_SK': [801, 802, 803, 804, 805],
    'is_vitals_missing': [0, 1, 0, 0, 1],
    'is_lab_outlier':    [0, 0, 1, 0, 0],
    'is_high_risk_flag': [0, 0, 1, 1, 0],
    'is_manual_entry':   [0, 1, 0, 0, 1],
    'is_transfer':       [0, 0, 0, 1, 1]
}

# 2. Build the DataFrame
df_quality = pd.DataFrame(data)

# 3. Export to Excel
df_quality.to_excel('Dim_Quality_Flags.xlsx', index=False, engine='openpyxl')

print("Success! Dim_Quality_Flags_BIT.xlsx generated with the BIT flag structure.")

Success! Dim_Quality_Flags_BIT.xlsx generated with the BIT flag structure.


## Generating Dim_Procedures Table

In [5]:
import pandas as pd

# 1. Define common CPT Procedures for the 2025 context
procedure_data = [
    {'SK': 1001, 'CPT': '99214', 'Desc': 'Office Visit - Level 4', 'Group': 'Evaluation', 'Cost': 150.00},
    {'SK': 1002, 'CPT': '93000', 'Desc': 'Electrocardiogram (EKG)', 'Group': 'Cardiology', 'Cost': 200.00},
    {'SK': 1003, 'CPT': '33533', 'Desc': 'CABG - Heart Bypass', 'Group': 'Surgery', 'Cost': 35000.00},
    {'SK': 1004, 'CPT': '71045', 'Desc': 'Chest X-Ray', 'Group': 'Radiology', 'Cost': 120.00},
    {'SK': 1005, 'CPT': '80053', 'Desc': 'Comprehensive Metabolic Panel', 'Group': 'Laboratory', 'Cost': 85.00},
    {'SK': 1006, 'CPT': '92920', 'Desc': 'Coronary Angioplasty', 'Group': 'Cardiology', 'Cost': 12000.00},
    {'SK': 1007, 'CPT': 'J1885', 'Desc': 'Injection - Ketorolac Tromethamine', 'Group': 'Pharmacy', 'Cost': 45.00},
    {'SK': 1008, 'CPT': '94640', 'Desc': 'Inhalation Treatment - Acute', 'Group': 'Respiratory', 'Cost': 110.00}
]

# 2. Create DataFrame
df_procs = pd.DataFrame(procedure_data)

# Rename columns to match schema
df_procs.columns = ['Procedure_SK', 'CPT_Code', 'Procedure_Description', 'Procedure_Group', 'Average_Cost']

# 3. Export to Excel
df_procs.to_excel('Dim_Procedures.xlsx', index=False, engine='openpyxl')
print("Success! Dim_Procedures.xlsx generated.")

Success! Dim_Procedures.xlsx generated.


## Generating Bridge_Procedures

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# 1. Define the specific local path for your Fact table
file_path = r'C:\Users\atifn\Downloads\Patient Re-Adm Risk Data\Fact_Encounters.xlsx'

# 2. Load your existing Fact table
if os.path.exists(file_path):
    df_fact = pd.read_excel(file_path)
    print(f"Successfully loaded Fact_Encounters from {file_path}")
else:
    print("Error: File not found at the specified location. Please check the path.")

# 3. Setup for the Bridge Table
np.random.seed(42)
bridge_records = []
# Using the Procedure_SKs from your Dim_Procedures (1001-1008)
procedure_pool = [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008]

# 4. Generate Clinical Data aligned with Fact Table dates
for index, row in df_fact.iterrows():
    enc_id = row['Encounter_SK']
    
    # Convert YYYYMMDD (handled as float/int) to clean string then to datetime
    admit_dt = str(int(row['Admission_Date_FK']))
    disch_dt = str(int(row['Discharge_Date_FK']))
    
    admit_obj = datetime.strptime(admit_dt, '%Y%m%d')
    disch_obj = datetime.strptime(disch_dt, '%Y%m%d')
    
    # Calculate Length of Stay (LOS)
    los = (disch_obj - admit_obj).days
    
    # Decide how many procedures (0-4) occurred during this specific stay
    num_procs = np.random.choice([0, 1, 2, 3, 4], p=[0.1, 0.5, 0.2, 0.1, 0.1])
    
    for _ in range(num_procs):
        # Pick a random day between Admit and Discharge
        random_day_offset = np.random.randint(0, los + 1) if los > 0 else 0
        proc_date_obj = admit_obj + timedelta(days=random_day_offset)
        
        bridge_records.append({
            'Encounter_FK': enc_id,
            'Procedure_FK': np.random.choice(procedure_pool),
            'Procedure_Date_FK': int(proc_date_obj.strftime('%Y%m%d')),
            # Sequence will be assigned after sorting below
            'Procedure_Sequence': 0 
        })

# 5. Transform to DataFrame and Apply Sequencing Logic
df_bridge = pd.DataFrame(bridge_records)

# ACTION: Sort by Encounter then Date to ensure chronological order
df_bridge = df_bridge.sort_values(by=['Encounter_FK', 'Procedure_Date_FK'])

# ACTION: Assign Procedure_Sequence (1, 2, 3...) based on the sorted dates
df_bridge['Procedure_Sequence'] = df_bridge.groupby('Encounter_FK').cumcount() + 1

# 6. Reorder columns for professional appearance
df_bridge = df_bridge[['Encounter_FK', 'Procedure_FK', 'Procedure_Sequence', 'Procedure_Date_FK']]

# 7. Export to the same folder
output_path = r'C:\Users\atifn\Downloads\Patient Re-Adm Risk Data\Bridge_Procedures.xlsx'
df_bridge.to_excel(output_path, index=False)

print(f"\nSuccess! Final Bridge Table generated and saved to: {output_path}")
print("Logic Applied: All procedures are now chronologically sequenced (Seq 1 is the earliest date).")

Successfully loaded Fact_Encounters from C:\Users\atifn\Downloads\Patient Re-Adm Risk Data\Fact_Encounters.xlsx

Success! Final Bridge Table generated and saved to: C:\Users\atifn\Downloads\Patient Re-Adm Risk Data\Bridge_Procedures_Final.xlsx
Logic Applied: All procedures are now chronologically sequenced (Seq 1 is the earliest date).
